# Branch and Bound Explorer: You Choose Where to Branch
**IN2040 · Optimization of Organizational Processes · Tecnológico de Monterrey** (Prof. Vincent Boyer)

*Session 7 · about 35 min.*

> **How to use this notebook.** Run the cells from top to bottom (Shift+Enter).
> Cells marked **✍️ TODO** are for you; a **✅ check** right after tells you if your answer is correct.
> Cells marked **🤖 AI prompt** contain a prompt you can paste into Gemini (the ✨ button in Colab),
> ChatGPT, Claude… **Always read and test the code the AI gives you.**

**You write the integer model in PuLP and you choose the branching variables. The computer solves the LP of
every node, closes the nodes it can, and draws the tree.** Models are kept small (at most 5 variables).

| Part | Topic | Time |
|------|-------|------|
| 1 | Example, then exercise: two variables (tree + picture) | 15 min |
| 2 | Example, then exercise: three variables, a minimization | 15 min |
| 3 | Does the order matter? Best-first vs depth-first | 5 min |

**Branch and bound** (maximization):
1. **Select** an open node: **best-first**, the one with the largest LP bound (on a tie, the newest node).
2. **Bound:** its LP relaxation gives $z_{LP}$ (solved when the node is created).
3. **Test:** LP infeasible → close · $z_{LP} \le z_{inc}$ → close **by bound** · integer solution → new
   **incumbent**, close · otherwise → **branch**.
4. **Branch** on a fractional $x_j = f$: two children $x_j \le \lfloor f \rfloor$ and $x_j \ge \lceil f \rceil$.
5. **Stop** when no open node is left: the incumbent is optimal.

Minimization: the LP value is a **lower** bound, and a node is closed by bound when $z_{LP} \ge z_{inc}$.

**How to drive it.** `branch_and_bound(model, branch_on)` takes your **ordered list** of branching variables:
the first name is used at the first node that has to be branched, the second at the next one, … Start with
`[]`, run, read which node is next and which variables are fractional there, add one name, run again… until
✅. You get the step table (incumbent, best bound, gap), the tree, and for 2 variables the LP point of every node.
Colours: blue = branched, green = integer, grey = closed by bound, red = infeasible, orange dashed = next.

In [ ]:
# @title Setup and the branch-and-bound explorer (run this cell; no need to read it)
import importlib, subprocess, sys
import importlib.metadata
try:                                   # PuLP 4 + its CBC solver (pulp[cbc]); replaces an older PuLP 3
    _old = int(importlib.metadata.version("pulp").split(".")[0]) < 4 or importlib.util.find_spec("cbcbox") is None
except importlib.metadata.PackageNotFoundError:
    _old = True
if _old:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pulp[cbc]>=4"], check=True)
import base64, json, math
import numpy as np, pandas as pd, matplotlib.pyplot as plt, pulp
from IPython.display import display, Markdown

_ANS = json.loads(base64.b64decode("eyJCX3oiOiA0MCwgIkJfeDEiOiA1LCAiQl94MiI6IDAsICJCX25vZGVzIjogNywgIk1feiI6IDIwLCAiTV94MSI6IDEsICJNX3gyIjogMiwgIk1feDMiOiAxLCAiTV9ub2RlcyI6IDl9"))

def check(key, value):
    """Compare your answer with the expected one."""
    if value is None:
        print(f"⏳ {key}: not answered yet"); return
    exp = _ANS[key]
    ok = abs(float(value) - exp) <= 1e-6 * max(1.0, abs(exp))
    print(f"✅ {key}: correct" if ok else f"❌ {key}: not yet")

MAX_VARS, MAX_ROWS, MAX_NODES = 5, 6, 60
COLORS = {"branched": ("#D0E8F8", "#0072CE"), "integer": ("#D5EFDF", "#007A4C"), "bound": ("#ECECEC", "#5b6b7c"),
          "infeasible": ("#F8D7DA", "#C8102E"), "open": ("#FFFFFF", "#5b6b7c"), "next": ("#FFF1DC", "#E07B00")}
STATUS_TEXT = {"branched": "branched", "integer": "INTEGER", "bound": "CLOSED BY BOUND", "infeasible": "INFEASIBLE",
               "open": "open", "next": "next: branch me"}


def fmt(v):
    """2.2222 -> '2.22', 6.0000001 -> '6'."""
    return str(int(round(v))) if abs(v - round(v)) < 1e-6 else f"{v:.2f}"


def check_model(model):
    xs, cons = model.variables(), model.constraints()
    if not xs or not cons or not model.objective or all(model.objective.get(v, 0) == 0 for v in xs):
        raise ValueError("The model has no objective or no constraints yet: did you fill in the ✍️ TODO lines?")
    if len(xs) > MAX_VARS or len(cons) > MAX_ROWS:
        raise ValueError(f"This explorer is for small models: at most {MAX_VARS} variables and {MAX_ROWS} constraints "
                         f"(yours has {len(xs)} and {len(cons)}).")
    ints = [v.name for v in xs if v.cat == pulp.LpInteger]
    if not ints:
        raise ValueError('No integer variable: declare them with cat="Integer" in add_variable.')
    return ints


def solve_lp(model, bounds):
    """LP relaxation of the model with extra bounds {name: (low, up)}; returns (x, z) or (None, None) if infeasible."""
    saved = {v.name: (v.lowBound, v.upBound, v.cat) for v in model.variables()}
    try:
        for v in model.variables():
            v.cat = pulp.LpContinuous
            lo, up = bounds.get(v.name, (None, None))
            if lo is not None:
                v.lowBound = lo if v.lowBound is None else max(v.lowBound, lo)
            if up is not None:
                v.upBound = up if v.upBound is None else min(v.upBound, up)
        result = model.solve(pulp.COIN_CMD(msg=False))
        if result.status_str == "Unbounded":
            raise ValueError("The LP relaxation is unbounded: add the missing constraints.")
        if result.status_str != "Optimal":
            return None, None
        return {v.name: v.value() for v in model.variables()}, pulp.value(model.objective)
    finally:
        for v in model.variables():
            v.lowBound, v.upBound, v.cat = saved[v.name]


def branch_and_bound(model, branch_on=(), rule="best", plot=True):
    """Branch-and-bound on a small PuLP integer model; YOU choose the branching variables.
    branch_on: ordered list of variable names, one per node that has to be branched (in the order the nodes are
               selected). When the list is used up, the explorer stops and tells you which node comes next.
    rule: "best" (best-first: the open node with the best LP bound; on a tie, the newest node) or
          "depth" (depth-first: the newest node, the ≤ child first)."""
    try:
        ints = check_model(model)
    except ValueError as err:
        display(Markdown(f"⚠️ **{err}**")); return None
    sign = 1 if model.sense == pulp.LpMaximize else -1          # compare sign * z: larger is better
    nodes, steps, stack = [], [], []
    inc = {"z": None, "x": None, "node": None}

    def create(parent, bounds, label):
        """Solve the node's LP; an infeasible node is closed by the caller (so the parent's step comes first)."""
        x, z = solve_lp(model, bounds)
        n = dict(id=len(nodes), parent=parent, label=label, bounds=bounds, x=x, z=z, step=None,
                 depth=0 if parent is None else nodes[parent]["depth"] + 1, status="open" if x is not None else "infeasible")
        nodes.append(n)
        if x is not None:
            stack.append(n["id"])
        return n

    def best_bound():
        vals = [sign * n["z"] for n in nodes if n["status"] == "open"] + ([sign * inc["z"]] if inc["z"] is not None else [])
        return None if not vals else sign * max(vals)

    def close(n, status, decision):
        n["status"], n["step"] = status, len(steps) + 1
        bnd = best_bound()
        gap = "" if inc["z"] is None or bnd is None else f"{100 * abs(bnd - inc['z']) / max(abs(inc['z']), 1e-9):.1f}%"
        steps.append({"Step": n["step"], "Node (branch)": f"N{n['id']} ({n['label']})",
                      "LP solution": "—" if n["x"] is None else "(" + ", ".join(fmt(n["x"][k]) for k in n["x"]) + ")",
                      "z_LP": "—" if n["z"] is None else fmt(n["z"]), "Decision": decision,
                      "z_inc": "—" if inc["z"] is None else fmt(inc["z"]),
                      "best bound": "—" if bnd is None else fmt(bnd), "gap": gap})

    root = create(None, {}, "root")
    if root["x"] is None:
        close(root, "infeasible", "LP infeasible: close")
    used, state, message, cand = 0, "done", "", []
    while stack:
        if len(nodes) > MAX_NODES:
            message = f"Stopped after {MAX_NODES} nodes."; state = "stopped"; break
        if rule == "best":
            k = max(stack, key=lambda i: (sign * nodes[i]["z"], i))
        else:
            k = stack[-1]
        n = nodes[k]
        x, z = n["x"], n["z"]
        if inc["z"] is not None and sign * z <= sign * inc["z"] + 1e-9:
            stack.remove(k)
            close(n, "bound", f"z_LP = {fmt(z)} {'≤' if sign > 0 else '≥'} z_inc = {fmt(inc['z'])}: close by bound")
            continue
        frac = [v for v in ints if abs(x[v] - round(x[v])) > 1e-6]
        if not frac:
            stack.remove(k)
            inc.update(z=z, x=x, node=k)
            close(n, "integer", "integer: new incumbent")
            continue
        if used >= len(branch_on):
            n["status"] = "next"; state = "waiting"; cand = frac
            why = "has the best bound" if rule == "best" else "is the newest open node (depth-first)"
            message = (f"**Your turn:** node N{k} ({n['label']}) {why} and is fractional: "
                       + ", ".join(f"{v} = {fmt(x[v])}" for v in frac)
                       + f". Add one of {frac} to your list, then run the cell again.")
            break
        name = branch_on[used]
        if name not in x:
            n["status"] = "next"; state = "stopped"; cand = frac
            message = f"⚠️ **{name!r} is not a variable of the model.** Fractional at N{k}: {frac}."; break
        if name not in frac:
            n["status"] = "next"; state = "stopped"; cand = frac
            why = "is not an integer variable" if name not in ints else f"= {fmt(x[name])} is already integer at N{k}"
            message = (f"⚠️ **{name} {why}**: branching on it would not remove the fractional point. "
                       f"Choose one of {frac}."); break
        used += 1
        stack.remove(k)
        f = math.floor(x[name] + 1e-9)
        lo, up = n["bounds"].get(name, (None, None))
        kids = [({**n["bounds"], name: (lo, f)}, f"{name} ≤ {f}"), ({**n["bounds"], name: (f + 1, up)}, f"{name} ≥ {f + 1}")]
        made = [create(k, b, lab) for b, lab in kids]
        close(n, "branched", f"fractional: branch on {name} = {fmt(x[name])}")
        for c in made:
            if c["x"] is None:
                close(c, "infeasible", "LP infeasible: close")
        if rule == "depth":            # depth-first: the ≤ child is explored first (top of the stack)
            for c in reversed(made):
                if c["id"] in stack:
                    stack.remove(c["id"]); stack.append(c["id"])
    # ---- report
    if steps:
        display(pd.DataFrame(steps).set_index("Step"))
    draw_tree(nodes, inc["node"], model.name)
    if state == "done":
        if inc["z"] is None:
            display(Markdown("❌ **No open node left and no integer solution: the integer problem is infeasible.**"))
        else:
            display(Markdown(f"✅ **No open node left: the incumbent is optimal.** "
                             + ", ".join(f"{k} = {fmt(v)}" for k, v in inc["x"].items())
                             + f", z = {fmt(inc['z'])}  ({len(nodes)} nodes, {len(steps)} steps)."))
    else:
        display(Markdown(message))
    if state == "done" and used < len(branch_on):
        display(Markdown(f"(The tree was finished before your list: {list(branch_on[used:])} not used.)"))
    if plot and len(model.variables()) == 2:
        plot_nodes(model, nodes, inc["node"])
    return dict(status=state, x=inc["x"], z=inc["z"], nodes=len(nodes), steps=len(steps), candidates=cand)


def draw_tree(nodes, best, title):
    """The tree: one box per node (LP solution and z), coloured by status; the incumbent has a thick border."""
    kids = {n["id"]: [m["id"] for m in nodes if m["parent"] == n["id"]] for n in nodes}
    pos, leaf = {}, [0]
    def place(k):
        if not kids[k]:
            pos[k] = leaf[0]; leaf[0] += 1
        else:
            for c in kids[k]:
                place(c)
            pos[k] = sum(pos[c] for c in kids[k]) / len(kids[k])
    place(0)
    depth = max(n["depth"] for n in nodes)
    fig, ax = plt.subplots(figsize=(max(5, 2.2 * leaf[0]), 1.9 * (depth + 1)))
    for n in nodes:
        X, Y = pos[n["id"]], -1.9 * n["depth"]
        if n["parent"] is not None:
            p = nodes[n["parent"]]
            PX, PY = pos[p["id"]], -1.9 * p["depth"]
            ax.plot([PX, X], [PY - 0.55, Y + 0.55], color="#9aa7b4", lw=1.2, zorder=1)
            ax.text((PX + X) / 2 + (0.08 if X > PX else -0.08), (PY + Y) / 2, n["label"], fontsize=8.5,
                    ha="left" if X > PX else "right", va="center", color="#1d2733")
        head = "Root (N0)" if n["parent"] is None else f"N{n['id']}"
        body = "LP infeasible" if n["x"] is None else "\n".join(
            [f"{k} = {fmt(v)}" for k, v in n["x"].items()] + [f"z = {fmt(n['z'])}"])
        step = "" if n["step"] is None else f"   ·  step {n['step']}"
        fc, ec = COLORS[n["status"]]
        ax.text(X, Y, f"{head}{step}\n{body}", ha="center", va="center", fontsize=8, zorder=2,
                bbox=dict(boxstyle="round,pad=0.4", fc=fc, ec=ec, lw=2.6 if n["id"] == best else 1.2,
                          ls="--" if n["status"] in ("open", "next") else "-"))
        if n["status"] != "branched":
            ax.text(X, Y - 0.5 - 0.08 * len(n["x"] or []), STATUS_TEXT[n["status"]], ha="center", va="top",
                    fontsize=7.5, fontweight="bold", color=ec)
    ax.set_xlim(-0.8, leaf[0] - 0.2); ax.set_ylim(-1.9 * depth - 1.4, 0.9); ax.axis("off")
    ax.set_title(f"{title}: {len(nodes)} nodes", fontsize=11)
    plt.show()


def plot_nodes(model, nodes, best):
    """2 variables: feasible region of the LP relaxation, integer points, and the LP optimum of every node."""
    xs = model.variables()
    rows = []
    for con in model.constraints():
        a, b = [con.get(v, 0) for v in xs], -con.constant
        rows.append((a, con.sense, b))
    pts = [n["x"] for n in nodes if n["x"] is not None]
    hi = 1.15 * max([max(abs(p[v.name]) for v in xs) for p in pts] + [1])
    X, Y = np.meshgrid(np.linspace(0, hi, 500), np.linspace(0, hi, 500))
    ok = np.ones_like(X, bool)
    for a, s, b in rows:
        lhs = a[0] * X + a[1] * Y
        ok &= (lhs <= b + 1e-9) if s == pulp.LpConstraintLE else (lhs >= b - 1e-9) if s == pulp.LpConstraintGE \
            else np.isclose(lhs, b, atol=hi / 250)
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    ax.contourf(X, Y, ok, levels=[0.5, 1], colors=["#0072CE"], alpha=0.15)
    grid = [(i, j) for i in range(int(hi) + 1) for j in range(int(hi) + 1)
            if all((a[0] * i + a[1] * j <= b + 1e-9) if s == pulp.LpConstraintLE else
                   (a[0] * i + a[1] * j >= b - 1e-9) if s == pulp.LpConstraintGE else
                   abs(a[0] * i + a[1] * j - b) < 1e-9 for a, s, b in rows)]
    if grid:
        ax.scatter(*zip(*grid), s=12, color="k", zorder=3, label="integer points")
    for n in nodes:
        if n["x"] is None:
            continue
        px, py = n["x"][xs[0].name], n["x"][xs[1].name]
        ax.scatter([px], [py], s=150 if n["id"] == best else 60, color=COLORS[n["status"]][1], zorder=4,
                   marker="*" if n["id"] == best else "o", edgecolor="white")
        ax.annotate(f"N{n['id']}", (px, py), textcoords="offset points", xytext=(5, 5), fontsize=8)
    ax.set_xlim(-0.2, hi); ax.set_ylim(-0.2, hi); ax.set_xlabel(xs[0].name); ax.set_ylabel(xs[1].name)
    ax.set_title("LP optimum of each node (★ = incumbent)", fontsize=10); ax.legend(fontsize=8, loc="upper right")
    plt.show()
print("Setup done ✔")

---
## 1 · Two variables
### Example
$\max\ z = 10x_1 + 15x_2$ s.t. $8x_1 + 4x_2 \le 40$, $15x_1 + 30x_2 \le 200$, $x_1, x_2 \ge 0$ **integer**.

In PuLP the only change from an LP is `cat="Integer"`:

In [ ]:
E = pulp.LpProblem("Example", pulp.LpMaximize)
x1 = E.add_variable("x1", lowBound=0, cat="Integer")
x2 = E.add_variable("x2", lowBound=0, cat="Integer")
E += 10 * x1 + 15 * x2
E += 8 * x1 + 4 * x2 <= 40, "c1"
E += 15 * x1 + 30 * x2 <= 200, "c2"
print(E)

Build the list one name at a time:
* `[]`: the root LP is $(2.22, 5.56)$, $z = 105.56$: both variables are fractional. Choose $x_2$.
* `["x2"]`: the children are $x_2 \le 5$ (N1, $z = 100$) and $x_2 \ge 6$ (N2, $z = 103.33$). Best-first takes N2,
  where only $x_1 = 1.33$ is fractional.
* `["x2", "x1"]`: N4 ($x_1 \ge 2$) is infeasible; N3 ($x_1 \le 1$) has $x_2 = 6.17$.
* `["x2", "x1", "x2"]`: N5 is integer (incumbent 100), N1 is closed by bound: ✅.

In [ ]:
E_branch = ["x2", "x1", "x2"]
resE = branch_and_bound(E, E_branch)

**Things to notice.**
* 7 nodes instead of enumerating every integer point; the optimum is $(1, 6)$, $z = 100$.
* N1 ($z_{LP} = 100$) is never branched: its bound cannot beat the incumbent 100. The gap is then 0%.
* In the picture, the root LP point is cut away by the branches, and **no integer point is lost**.
* Try `E_branch = ["x1", ...]`: another tree (9 nodes), **the same optimum**. The choice changes the work, not the answer.

### ✍️ Exercise 1
$$\max\ z = 8x_1 + 5x_2 \quad \text{s.t.}\quad x_1 + x_2 \le 6,\;\; 9x_1 + 5x_2 \le 45,\;\; x_1, x_2 \ge 0 \text{ integer}$$
1. Write the model in PuLP (variables `x1`, `x2`, integer).
2. Run B&B branching each time on the **first** fractional variable ($x_1$ before $x_2$): build `B_branch`
   one name at a time.
3. The root LP point rounds to $(4, 2)$ or $(3, 2)$. Is $(4, 2)$ feasible? How far is the integer optimum from them?

In [ ]:
B = pulp.LpProblem("Exercise 1", pulp.LpMaximize)
x1 = B.add_variable("x1", lowBound=0, cat="Integer")
x2 = B.add_variable("x2", lowBound=0, cat="Integer")
# ✍️ TODO: the objective and the two constraints

B_branch = []            # ✍️ TODO: add the branching variables one at a time, e.g. ["x1"], then ["x1", ...]
resB = branch_and_bound(B, B_branch)

In [ ]:
B_z = None               # ✍️ optimal z of the integer problem
B_x1 = None              # ✍️ optimal x1
B_x2 = None              # ✍️ optimal x2
B_nodes = None           # ✍️ number of nodes of your tree (first-fractional rule)
for k in ["B_z", "B_x1", "B_x2", "B_nodes"]:
    check(k, globals()[k])

---
## 2 · Three variables, a minimization
No picture any more: the tree and the step table are all you have. For a **minimization**, the LP value of a
node is a **lower** bound on what its subtree can reach, and best-first takes the open node with the
**smallest** LP value.

### ✍️ Exercise 2
A kitchen buys packs of three types to cover three daily needs. Packs must be whole.
$$\min\ z = 4x_1 + 5x_2 + 6x_3 \quad \text{s.t.}\quad 2x_1 + x_2 + 3x_3 \ge 7,\;\; x_1 + 3x_2 + 2x_3 \ge 8,\;\;
3x_1 + 2x_2 + x_3 \ge 6,\;\; x_1, x_2, x_3 \ge 0 \text{ integer}$$
1. Write the model in PuLP and run B&B branching on the **first** fractional variable each time.
2. **Challenge:** find another list that closes the tree with **fewer nodes**. Same optimum?

In [ ]:
M = pulp.LpProblem("Exercise 2", pulp.LpMinimize)
x1 = M.add_variable("x1", lowBound=0, cat="Integer")
x2 = M.add_variable("x2", lowBound=0, cat="Integer")
x3 = M.add_variable("x3", lowBound=0, cat="Integer")
# ✍️ TODO: the objective and the three constraints

M_branch = []            # ✍️ TODO: add the branching variables one at a time
resM = branch_and_bound(M, M_branch)

In [ ]:
M_z = None               # ✍️ optimal z
M_x1 = None              # ✍️ optimal x1
M_x2 = None              # ✍️ optimal x2
M_x3 = None              # ✍️ optimal x3
M_nodes = None           # ✍️ number of nodes with the first-fractional rule
for k in ["M_z", "M_x1", "M_x2", "M_x3", "M_nodes"]:
    check(k, globals()[k])

---
## 3 · Does the node order matter?
`rule="depth"` explores **depth-first**: always the newest node, the $\le$ child first. It finds an integer
solution quickly, but may explore nodes that best-first would close by bound. Run the example with it
(it asks for a new list, since the nodes come in another order):

In [ ]:
E_depth = ["x2"]         # ✍️ continue the list until ✅
branch_and_bound(E, E_depth, rule="depth");

### Take-aways
* Every node is an **LP**; branching removes a strip with no integer point, so **no integer solution is lost**.
* A node is closed when its LP is infeasible, integer, or **not better than the incumbent** (bound).
* The **branching variable** and the **node order** change the size of the tree, never the optimum.
* Incumbent and best bound sandwich the optimum: gap $= 0$ means proven optimal. Real solvers stop at a
  small gap or a time limit.

### 🚀 If you have time
* Rounding the root LP of Exercise 1 gives an infeasible or poor point. Solve the IP with
  `B.solve(pulp.COIN_CMD(msg=False))` and compare with your tree.
* Write a small integer model of your own (at most 5 variables) and explore it.